# Simulation des résultats du sondage sur l'efficacité du Pass Culture

Dans ce notebook, nous allons simuler les résultats du sondage mené auprès des utilisateurs du Pass Culture afin d'en évaluer l'efficacité.

Les bases de données utilisées ci-après ont été récupérées auprès de la Cour des Comptes : https://www.ccomptes.fr/fr/publications/premier-bilan-du-pass-culture. 

Tout d'abord, nous devons impoorter les éléments nécessaires.

In [13]:
from functions import *
import pandas as pd

## Importation des données

Dans un premier temps, nous importons les 16 bases de données mises à disposition par la Cour des Comptes. Elles regroupent des statistiques agrégées et nous pouvons ainsi reproduire les figures présentées dans le rapport d'évaluation de la Cour.

In [39]:
dfs = {}  # dictionnaire pour stocker tous les DataFrames
for elt in ["C3", "C4", "G1", "G2", "G3", "G4", "G5", "G6", "G7", "G8", "G9", "G10", "G11", "G13", "G14", "G15"]:  # liste des df que l'on veut importer
    path = elt+".csv"
    dfs[f"df_{elt.lower()}"] = pd.read_csv(path, sep=";", encoding="latin-1") 
    # On ne peut pas utiliser importdata car l'encodage du fichier n'est pas le même

Nous avons donc obtenu 16 bases de données qui reprennent des statistiques descriptives sur le sondage à partir duquel le dispositif a été évalué. Nous devons maintenant simuler les données à partir de ces statistiques agrégées.

## Description des données à notre disposition

Dans un premier temps, nous allons récapituler les données dont nous disposons, puis nous passerons à la simulation. Pour créer le tableau suivant, nous avons utilisé la fonction "infosbase" sur chaque dataframe. 

<table>
  <caption>
    Données mises à disposition par la Cour des Comptes
  </caption>
  <thead>
    <tr>
      <th scope="col">Nom du data_frame</th>
      <th scope="col">Variables</th>
      <th scope="col">Nombre de lignes</th>
      <th scope="col">Description</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th scope="row">df_c3</th>
      <td>Département<br>Montant.moyen.dépensé.par.les.jeunes.du.département</td>
      <td>102 lignes</td>
      <td>Donne le montant moyen dépensé par les jeunes du département.</td>
    </tr>
    <tr>
      <th scope="row">df_c4</th>
      <td>Département<br>Score.de.diversification.moyen.par.département</td>
      <td>102 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g1</th>
      <td>Âge<br>15.+<br>16.+<br>17.+<br>18.+</td>
      <td>1 ligne</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g2</th>
      <td>Caractéristique<br>Valeur<br>Taux.d'activitation.du.pass</td>
      <td>12 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g3</th>
      <td>Statut.déclaré<br>Etudiant<br>Lycéen<br>Collégien<br>Apprenti,.alternant,.service.civique<br>Demandeur.d'emploi<br>Employé<br>Inactif</td>
      <td>1 ligne</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g4</th>
      <td>Trimestre<br>cat_agrr<br>prop_aggr</td>
      <td>66 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g5</th>
      <td>categories<br>Population<br>Pourcentage</td>
      <td>20 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g6</th>
      <td>trimestre<br>macro_rayon_r<br>prop_montant</td>
      <td>99 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g7</th>
      <td>categories<br>nombre_utilisateurs<br>Revenu</td>
      <td>42 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g8</th>
      <td>trimestre<br>Age.à.la.réservation<br>home<br>search</td>
      <td>20 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g9</th>
      <td>Origin<br>Composante.diversité<br>Prop</td>
      <td>10 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g10</th>
      <td>Origine<br>Catégorie<br>prop</td>
      <td>12 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g11</th>
      <td>Delta.de.diversification<br>Part.des.réservations</td>
      <td>6 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g13</th>
      <td>X1<br>LFI.2022<br>Exec.2022<br>LFI.2023<br>Exec.2023<br>LFI.2024<br>Exec.(prév.).2024</td>
      <td>2 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g14</th>
      <td>Service<br>Effectif</td>
      <td>6 lignes</td>
      <td></td>
    </tr>
    <tr>
      <th scope="row">df_g15</th>
      <td>X1<br>16.ans<br>17.ans<br>18.ans<br>19.ans</td>
      <td>4 lignes</td>
      <td></td>
    </tr>
  </tbody>
</table>


## Simulation 

Objectif :

Simuler une base de données au niveau individuel à partir de statistiques agrégées (moyennes, proportions, tableaux croisés)
publiées dans un rapport (ici : Cour des comptes, Pass Culture).
 
Structure du script :

1. Fondements théoriques : explication de la théorie mathématique sur laquelle se fonde la simulation, justification du choix de la méthode ;

2. Configuration générale (graine aléatoire, taille de l'échantillon)

3. Fonction d'ajustement proportionnel itératif (IPF / raking)
       -> pour caler une table jointe sur plusieurs marges connues

4. Fonction de validation : on ré-agrège les données simulées et on
       les compare aux statistiques d'origine

Les fonctions à créer seront à nouveaux définies dans le fichier functions.py afin de fluidifier la lecture du notebook.

### Fondements théoriques

#### Problème : inférence écologique

La désagrégation statistique répond à un problème d'inférence statistique, ou inférence écologique. On dispose de données agrégées (moyennes, pourcentages, totaux par groupes), et on souhaite reconstituer une base de données au niveau individuel, alors qu'on ne dispose pas des données à ce niveau.

L'objectif n'est ainsi pas de retrouver les "vraies" données, mais de simuler une base de données qui aurait mené aux mêmes résultats agrégés. Nous sommes donc bien dans une situation de simulation sous contrainte, et non de reconstruction.

L'inférence est ici dite "stochastique car la désagrégation repose sur des tirages aléatoires (type Monte Carlo <mark>à développer / préciser</mark>), plutôt que sur une règle déterministe. La valeur prise par une certaine variable est tirée aléatoirement pour chaque individu, afin de conserver l'aléa et la variabilité naturels que l'on observerait dans un véritable échantillon.

#### Difficultés principales 

La difficulté principale pour notre simulation est que les lois marginales ne permettent pas de déduire la loi jointe. 

Dans certains cas, l'hypothèse la plus simple est de supposer l'indépendance des variables. Chaque variable sera alors simulée séparément, indépendamment des autres. Cette solution est rapide est simple, mais elle peut introduire un biais lorsqu'en réalité, les variables sont corrélées.

Lorsque nous disposons de tableaux qui croisent déjà plusieurs variables, nous pourrons nous passer de l'hypothèse d'indépendance. En effet, nous avons alors des informations sur la loi jointe des différentes variables proposées, et nous pourrons donc les simuler ensemble au lieu de les simuler indépendamment. Cette option sera préférable lorsqu'elle est possible.

Ces deux options sont les deux pôles à partir desquels nous allons effectuer la simulation. 


### Configuration générale

La configuration est une étape importante car elle garantit la reproductibilité des résultats.

In [ ]:
seed = 42                # graine aléatoire -> reproductibilité des simulations
nb_indiv = 10_000      # taille de l'échantillon simulé (à ajuster)
 
rng = np.random.default_rng(seed)

### Fonction d'ajustement proportionnel

### Validation